misc notes
- response needs to have user idx, item idx, and relevance

# set up environment

In [2]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
import shutil

from pyspark.sql import SparkSession
from pyspark.sql import functions as sf
from pyspark.sql import DataFrame, Window
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.linalg import Vectors, VectorUDT
from pyspark.sql.types import DoubleType, ArrayType

# Set up plotting
plt.style.use('seaborn-whitegrid')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)


/home/morgan/miniconda3/envs/venv39/lib/python3.9/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
/tmp/ipykernel_1799/45819036.py:18: MatplotlibDeprecationWarning: The seaborn styles shipped by Matplotlib are deprecated since 3.6, as they no longer correspond to the styles shipped by seaborn. However, they will remain available as 'seaborn-v0_8-<style>'. Alternatively, directly use the seaborn API instead.
  plt.style.use('seaborn-whitegrid')


# init spark session 

In [3]:
spark = (SparkSession.builder
    .appName("RecSysVisualization")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print("Spark session initialized.")

your 131072x1 screen size is bogus. expect trouble
25/05/08 18:51:08 WARN Utils: Your hostname, MorganTP resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/05/08 18:51:08 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/05/08 18:51:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark session initialized.


In [4]:
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), '..'))

from data_generator import CompetitionDataGenerator
from simulator import CompetitionSimulator
from sample_recommenders import (
    RandomRecommender,
    PopularityRecommender,
    ContentBasedRecommender
)
from config import DEFAULT_CONFIG, EVALUATION_METRICS

print("Competition modules imported.")

Competition modules imported.


/home/morgan/miniconda3/envs/venv39/lib/python3.9/site-packages/rdt/transformers/base.py:132: FutureWarning: Future versions of RDT will not support the 'model_missing_values' parameter. Please switch to using the 'missing_value_generation' parameter to select your strategy.
  warnings.warn(


In [5]:
from pyspark.ml.classification import LogisticRegression 
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder, StandardScaler
from pyspark.ml import Pipeline, param



class MyRecommender:
    """
    Template class for implementing a custom recommender.
    
    This class provides the basic structure required to implement a recommender
    that can be used with the Sim4Rec simulator. Students should extend this class
    with their own recommendation algorithm.
    """
    
    def __init__(self, seed=None):
        """
        Initialize recommender.
        
        Args:
            seed: Random seed for reproducibility
        """
        self.seed = seed
        # Add your initialization logic here
        self.model = None
    
    @staticmethod
    def one_hot_encode(df, column):
        pass

    def fit(self, log, user_features=None, item_features=None):
        """
        Train the recommender model based on interaction history.
        
        Args:
            log: Interaction log with user_idx, item_idx, and relevance columns
            user_features: User features dataframe (optional)
            item_features: Item features dataframe (optional)
        """
        # Implement your training logic here
        data = (log
                .join(user_features, on="user_idx", how="left")
                .join(item_features, on="item_idx", how="left")
                .drop("user_idx", "item_idx")
            )
                
    
    def predict(self, log, k, users, items, user_features=None, item_features=None, filter_seen_items=True):
        """
        Generate recommendations for users.
        
        Args:
            log: Interaction log with user_idx, item_idx, and relevance columns
            k: Number of items to recommend
            users: User dataframe
            items: Item dataframe
            user_features: User features dataframe (optional)
            item_features: Item features dataframe (optional)
            filter_seen_items: Whether to filter already seen items
        
        Returns:
            DataFrame: Recommendations with user_idx, item_idx, and relevance columns
        """
        # Example of a random recommender implementation:
        # Cross join users and items
        recs = users.crossJoin(items)
        
        # Filter out already seen items if needed
        if filter_seen_items and log is not None:
            seen_items = log.select("user_idx", "item_idx")
            recs = recs.join(
                seen_items,
                on=["user_idx", "item_idx"],
                how="left_anti"
            )
        
        # Add random relevance scores
        recs = recs.withColumn(
            "relevance",
            sf.rand(seed=self.seed)
        )
        
        # Rank items by relevance for each user
        window = Window.partitionBy("user_idx").orderBy(sf.desc("relevance"))
        recs = recs.withColumn("rank", sf.row_number().over(window))
        
        # Filter top-k recommendations
        recs = recs.filter(sf.col("rank") <= k).drop("rank")
        
        return recs

print("MyRecommender template defined.")

MyRecommender template defined.


In [6]:
# We'll create a smaller dataset for demonstration.
config = DEFAULT_CONFIG.copy()
config['data_generation']['n_users'] = 1000  # Reduced from 10,000
config['data_generation']['n_items'] = 200   # Reduced from 1,000
config['data_generation']['seed'] = 42       # Fixed seed for reproducibility

# Get train-test split parameters
train_iterations = config['simulation']['train_iterations']
test_iterations = config['simulation']['test_iterations']

print(f"Running train-test simulation with {train_iterations} training iterations and {test_iterations} testing iterations")

# Initialize data generator
data_generator = CompetitionDataGenerator(
    spark_session=spark,
    **config['data_generation']
)

# Generate user data
users_df = data_generator.generate_users()
print(f"Generated {users_df.count()} users")

# Generate item data
items_df = data_generator.generate_items()
print(f"Generated {items_df.count()} items")

# Generate initial interaction history
history_df = data_generator.generate_initial_history(
    config['data_generation']['initial_history_density']
)
print(f"Generated {history_df.count()} initial interactions")

Running train-test simulation with 5 training iterations and 5 testing iterations


25/05/08 18:51:14 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


Generated 1000 users
Generated 200 items
Generated 200 initial interactions


## exploring data

In [7]:
import pprint
pprint.pprint(config)

{'competition': {'evaluation_iterations': 20,
                 'evaluation_metric': 'revenue',
                 'max_submission_per_day': 3,
                 'private_leaderboard_fraction': 0.6,
                 'public_leaderboard_fraction': 0.4,
                 'time_limit_seconds': 3600},
 'data_generation': {'initial_history_density': 0.001,
                     'item_categories': ['electronics',
                                         'books',
                                         'clothing',
                                         'home'],
                     'item_category_means': [0.3, -0.3, 0.0, 0.2],
                     'item_category_weights': [0.2, 0.3, 0.3, 0.2],
                     'item_feature_mean': 0.0,
                     'item_feature_std': 1.0,
                     'n_item_features': 20,
                     'n_items': 200,
                     'n_user_features': 20,
                     'n_users': 1000,
                     'seed': 42,
                  

In [8]:
print(type(users_df))
users_df.select('user_idx').count() - len(users_df.select('user_idx').distinct().collect())
# user idx is user id
users_df.show(5)
users_df.groupBy('segment').count().show()
users_df.describe().show()
# user features already normalized (enough)

<class 'pyspark.sql.dataframe.DataFrame'>
+--------------------+-------------------+--------------------+-------------------+-------------------+--------------------+-------------------+--------------------+--------------------+--------------------+--------------------+-------------------+-------------------+-------------------+-------------------+-------------------+--------------------+--------------------+--------------------+-------------------+--------+-------+
|         user_attr_0|        user_attr_1|         user_attr_2|        user_attr_3|        user_attr_4|         user_attr_5|        user_attr_6|         user_attr_7|         user_attr_8|         user_attr_9|        user_attr_10|       user_attr_11|       user_attr_12|       user_attr_13|       user_attr_14|       user_attr_15|        user_attr_16|        user_attr_17|        user_attr_18|       user_attr_19|user_idx|segment|
+--------------------+-------------------+--------------------+-------------------+-----------------

25/05/08 18:51:21 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+--------------------+--------------------+-------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-------------------+--------------------+--------------------+--------------------+-----------------+-------+
|summary|         user_attr_0|         user_attr_1|        user_attr_2|         user_attr_3|         user_attr_4|         user_attr_5|         user_attr_6|         user_attr_7|         user_attr_8|         user_attr_9|        user_attr_10|        user_attr_11|        user_attr_12|        user_attr_13|        user_attr_14|        user_attr_15|       user_attr_16|        user_attr_17|        user_attr_18|        user_attr_19|         user_idx|segment|
+-------+--------------------+--------------------+-------------------+--------------------+

In [9]:
print(type(items_df))
items_df.show(5)
items_df.select('item_idx').count() - len(items_df.select('item_idx').distinct().collect())
# item idx is user id
items_df.groupBy('category').count().show()
items_df.describe().show()
# item_attr_.* are normalized, price is not

<class 'pyspark.sql.dataframe.DataFrame'>
+-------------------+--------------------+-------------------+--------------------+-------------------+--------------------+--------------------+-------------------+-------------------+--------------------+--------------------+--------------------+-------------------+-------------------+--------------------+-------------------+-------------------+--------------------+-------------------+-------------------+--------+-----------+------------------+
|        item_attr_0|         item_attr_1|        item_attr_2|         item_attr_3|        item_attr_4|         item_attr_5|         item_attr_6|        item_attr_7|        item_attr_8|         item_attr_9|        item_attr_10|        item_attr_11|       item_attr_12|       item_attr_13|        item_attr_14|       item_attr_15|       item_attr_16|        item_attr_17|       item_attr_18|       item_attr_19|item_idx|   category|             price|
+-------------------+--------------------+--------------

+-------+--------------------+--------------------+--------------------+--------------------+--------------------+-------------------+-------------------+--------------------+-------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-------------------+-------------------+--------------------+-------------------+--------------------+--------------------+------------------+--------+------------------+
|summary|         item_attr_0|         item_attr_1|         item_attr_2|         item_attr_3|         item_attr_4|        item_attr_5|        item_attr_6|         item_attr_7|        item_attr_8|         item_attr_9|        item_attr_10|        item_attr_11|        item_attr_12|        item_attr_13|       item_attr_14|       item_attr_15|        item_attr_16|       item_attr_17|        item_attr_18|        item_attr_19|          item_idx|category|             price|
+-------+--------------------+--------------------+-------

In [10]:
history_df.show(5)
history_df.groupBy('user_idx').count().groupBy('count').count().show()
# many one-time interactions
history_df.groupBy('item_idx').count().groupBy('count').count().show()
# a couple of popular items 

+--------+--------+---------+
|user_idx|item_idx|relevance|
+--------+--------+---------+
|     396|      97|        1|
|     336|      25|        0|
|     935|     102|        0|
|     967|     197|        0|
|     854|      96|        1|
+--------+--------+---------+
only showing top 5 rows

+-----+-----+
|count|count|
+-----+-----+
|    1|  162|
|    2|   19|
+-----+-----+

+-----+-----+
|count|count|
+-----+-----+
|    1|   81|
|    3|    4|
|    2|   41|
|    4|    5|
|    5|    1|
+-----+-----+



# expr

In [11]:
data = (
    history_df.join(users_df, on="user_idx", how="left")
    .join(items_df, on="item_idx", how="left")
    .drop("user_idx", "item_idx")
)
print(type(data), type(history_df), type(items_df), type(users_df))
data.show(5)


<class 'pyspark.sql.dataframe.DataFrame'> <class 'pyspark.sql.dataframe.DataFrame'> <class 'pyspark.sql.dataframe.DataFrame'> <class 'pyspark.sql.dataframe.DataFrame'>
+---------+-------------------+-------------------+--------------------+-------------------+--------------------+------------------+-------------------+--------------------+-------------------+-------------------+-------------------+-------------------+--------------------+--------------------+--------------------+-------------------+-------------------+-------------------+--------------------+--------------------+----------+-------------------+-------------------+--------------------+-------------------+-------------------+-------------------+-------------------+-------------------+--------------------+-------------------+-------------------+--------------------+-------------------+-------------------+--------------------+--------------------+-------------------+-------------------+-------------------+------------------

## pyspark logreg

In [12]:
import re


price_name  = "price"
data_pipeline = Pipeline(
    stages=[
        categoryIDX := StringIndexer(inputCol="category", outputCol="category_idx"),
        segmentIDX := StringIndexer(inputCol="segment", outputCol="segment_idx"),
        categoryEncode := OneHotEncoder(
            inputCol="category_idx", outputCol="category_vec"
        ),
        segmentEncode := OneHotEncoder(inputCol="segment_idx", outputCol="segment_vec"),
        price_assembler := VectorAssembler(
            inputCols=['price'], outputCol="pricev"
        ), 
        scaler:=StandardScaler(
            inputCol="pricev", outputCol="pricevs", withMean=True, withStd=True
        ),
        assembler := VectorAssembler(
            inputCols=
            [c for c in data.columns if re.match(r".*_attr_.*", c)] + 
                ["category_vec", "segment_vec", "pricevs"],
            outputCol=(feature_name:="features"),
        ),
    ]
)

model_pipeline = Pipeline(
    stages=[
        data_pipeline, 
        logistic := LogisticRegression(featuresCol=feature_name, labelCol="relevance")
    ]
)

In [14]:
data.columns

['relevance',
 'user_attr_0',
 'user_attr_1',
 'user_attr_2',
 'user_attr_3',
 'user_attr_4',
 'user_attr_5',
 'user_attr_6',
 'user_attr_7',
 'user_attr_8',
 'user_attr_9',
 'user_attr_10',
 'user_attr_11',
 'user_attr_12',
 'user_attr_13',
 'user_attr_14',
 'user_attr_15',
 'user_attr_16',
 'user_attr_17',
 'user_attr_18',
 'user_attr_19',
 'segment',
 'item_attr_0',
 'item_attr_1',
 'item_attr_2',
 'item_attr_3',
 'item_attr_4',
 'item_attr_5',
 'item_attr_6',
 'item_attr_7',
 'item_attr_8',
 'item_attr_9',
 'item_attr_10',
 'item_attr_11',
 'item_attr_12',
 'item_attr_13',
 'item_attr_14',
 'item_attr_15',
 'item_attr_16',
 'item_attr_17',
 'item_attr_18',
 'item_attr_19',
 'category',
 'price']

In [1]:
train_data, test_data = data.randomSplit([0.7, 0.3]) 
fit_model = model_pipeline.fit(train_data)
predictions = fit_model.transform(train_data)
predictions.show(5)

NameError: name 'data' is not defined

In [13]:
predictions.select('relevance rawPrediction probability prediction'.split()).show(10, truncate=False)

NameError: name 'predictions' is not defined

In [ ]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator 
res = BinaryClassificationEvaluator(
    rawPredictionCol='prediction',labelCol='relevance', metricName='areaUnderROC'
    ).evaluate(predictions) 
print(f"Area under ROC: {res}")

Area under ROC: 0.45769682726204464


## sklearn logreg

In [15]:
datadf = data.toPandas()
print(type(datadf))
datadf.head(5)
datadf = pd.get_dummies(datadf, columns=['category', 'segment'])
datadf.head(5)

<class 'pandas.core.frame.DataFrame'>


,relevance,user_attr_0,user_attr_1,user_attr_2,user_attr_3,user_attr_4,user_attr_5,user_attr_6,user_attr_7,user_attr_8,...,item_attr_18,item_attr_19,price,category_books,category_clothing,category_electronics,category_home,segment_budget,segment_mainstream,segment_premium
0,1,1.310606,1.171102,-1.256565,1.289203,-0.318159,0.600733,-0.906215,0.141980,0.313723,...,-1.364378,1.602430,27.295823,True,False,False,False,False,True,False
1,0,0.568273,1.086381,0.333370,-2.492810,-0.499139,1.197168,-0.807591,-0.223473,-1.044836,...,-0.734805,-1.288644,66.040801,False,False,True,False,False,True,False
2,0,-1.094826,1.812724,-0.636871,1.247187,-0.499947,1.817384,2.706783,-1.186530,-0.584756,...,0.013872,-0.558278,32.074983,False,True,False,False,False,False,True
3,0,0.177759,-0.812608,0.423006,1.764916,1.503137,0.731793,1.612268,-0.559519,0.174546,...,1.650976,-0.195459,17.036604,False,False,False,True,False,False,True
4,1,1.609099,0.395228,-0.013488,1.842748,-0.154446,1.129476,0.441578,-1.341566,1.628594,...,-1.109917,0.548874,47.274143,True,False,False,False,False,False,True


In [16]:
x = datadf.drop(columns=['relevance'])
y = datadf['relevance']
x.head(5)

,user_attr_0,user_attr_1,user_attr_2,user_attr_3,user_attr_4,user_attr_5,user_attr_6,user_attr_7,user_attr_8,user_attr_9,...,item_attr_18,item_attr_19,price,category_books,category_clothing,category_electronics,category_home,segment_budget,segment_mainstream,segment_premium
0,1.310606,1.171102,-1.256565,1.289203,-0.318159,0.600733,-0.906215,0.141980,0.313723,-1.484605,...,-1.364378,1.602430,27.295823,True,False,False,False,False,True,False
1,0.568273,1.086381,0.333370,-2.492810,-0.499139,1.197168,-0.807591,-0.223473,-1.044836,1.783923,...,-0.734805,-1.288644,66.040801,False,False,True,False,False,True,False
2,-1.094826,1.812724,-0.636871,1.247187,-0.499947,1.817384,2.706783,-1.186530,-0.584756,1.273132,...,0.013872,-0.558278,32.074983,False,True,False,False,False,False,True
3,0.177759,-0.812608,0.423006,1.764916,1.503137,0.731793,1.612268,-0.559519,0.174546,0.895805,...,1.650976,-0.195459,17.036604,False,False,False,True,False,False,True
4,1.609099,0.395228,-0.013488,1.842748,-0.154446,1.129476,0.441578,-1.341566,1.628594,-2.257854,...,-1.109917,0.548874,47.274143,True,False,False,False,False,False,True


In [17]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
x['price'] = scaler.fit_transform(x[['price']])
x.head()


,user_attr_0,user_attr_1,user_attr_2,user_attr_3,user_attr_4,user_attr_5,user_attr_6,user_attr_7,user_attr_8,user_attr_9,...,item_attr_18,item_attr_19,price,category_books,category_clothing,category_electronics,category_home,segment_budget,segment_mainstream,segment_premium
0,1.310606,1.171102,-1.256565,1.289203,-0.318159,0.600733,-0.906215,0.141980,0.313723,-1.484605,...,-1.364378,1.602430,-1.070549,True,False,False,False,False,True,False
1,0.568273,1.086381,0.333370,-2.492810,-0.499139,1.197168,-0.807591,-0.223473,-1.044836,1.783923,...,-0.734805,-1.288644,0.760421,False,False,True,False,False,True,False
2,-1.094826,1.812724,-0.636871,1.247187,-0.499947,1.817384,2.706783,-1.186530,-0.584756,1.273132,...,0.013872,-0.558278,-0.844700,False,True,False,False,False,False,True
3,0.177759,-0.812608,0.423006,1.764916,1.503137,0.731793,1.612268,-0.559519,0.174546,0.895805,...,1.650976,-0.195459,-1.555368,False,False,False,True,False,False,True
4,1.609099,0.395228,-0.013488,1.842748,-0.154446,1.129476,0.441578,-1.341566,1.628594,-2.257854,...,-1.109917,0.548874,-0.126434,True,False,False,False,False,False,True


In [18]:
from sklearn.linear_model import LogisticRegression
logistic = LogisticRegression()
logistic.fit(x, y)
predictions = logistic.predict(x)


In [19]:
from sklearn.metrics import roc_auc_score, confusion_matrix
roc_auc_score(y, predictions)

0.7043478260869566

In [20]:
confusion_matrix(y, predictions)

array([[93, 22],
       [34, 51]])

In [35]:
print(type(predictions))

<class 'numpy.ndarray'>


# setting up generators and recommenders

In [20]:
user_generator, item_generator = data_generator.setup_data_generators()


In [21]:
print(type(user_generator))

<class 'sim4rec.modules.generator.CompositeGenerator'>


In [ ]:

# Initialize the recommenders we want to compare
recommenders = [
    RandomRecommender(seed=42),
    PopularityRecommender(alpha=1.0, seed=42),
    ContentBasedRecommender(similarity_threshold=0.0, seed=42),
]
recommender_names = ["Random", "Popularity", "ContentBased", "MyRecommender"]

# Fit each recommender on the initial history
# for recommender in recommenders:
#     recommender.fit(log=data_generator.history_df)

print("Recommenders set up and initial fit complete.")

Recommenders set up and initial fit complete.


In [23]:
user_generator.sample(0.8).show(5)
item_generator.sample(0.8).show(5)

+--------------------+--------------------+-------------------+-------------------+--------------------+--------------------+--------------------+-------------------+-------------------+-------------------+--------------------+-------------------+--------------------+--------------------+-------------------+--------------------+--------------------+--------------------+--------------------+-------------------+--------+-------+
|         user_attr_0|         user_attr_1|        user_attr_2|        user_attr_3|         user_attr_4|         user_attr_5|         user_attr_6|        user_attr_7|        user_attr_8|        user_attr_9|        user_attr_10|       user_attr_11|        user_attr_12|        user_attr_13|       user_attr_14|        user_attr_15|        user_attr_16|        user_attr_17|        user_attr_18|       user_attr_19|user_idx|segment|
+--------------------+--------------------+-------------------+-------------------+--------------------+--------------------+-------------